<a href="https://colab.research.google.com/github/AchintyaGhayal/BA---AI---Project/blob/main/week2_sql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:




import pandas as pd
import sqlite3

url = "https://raw.githubusercontent.com/AchintyaGhayal/BA---AI---Project/refs/heads/main/coffee-shop-sales-revenue.csv"

df = pd.read_csv(url, sep="|")

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

df.head()




Dataset shape: (149116, 11)
Columns: ['transaction_id', 'transaction_date', 'transaction_time', 'transaction_qty', 'store_id', 'store_location', 'product_id', 'unit_price', 'product_category', 'product_type', 'product_detail']


,transaction_id,transaction_date,transaction_time,transaction_qty,store_id,store_location,product_id,unit_price,product_category,product_type,product_detail
0,1,2023-01-01,07:06:11,2,5,Lower Manhattan,32,3.0,Coffee,Gourmet brewed coffee,Ethiopia Rg
1,2,2023-01-01,07:08:56,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg
2,3,2023-01-01,07:14:04,2,5,Lower Manhattan,59,4.5,Drinking Chocolate,Hot chocolate,Dark chocolate Lg
3,4,2023-01-01,07:20:24,1,5,Lower Manhattan,22,2.0,Coffee,Drip coffee,Our Old Time Diner Blend Sm
4,5,2023-01-01,07:22:41,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg


In [6]:

df = df.drop_duplicates().dropna()

df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

df["transaction_qty"] = pd.to_numeric(df["transaction_qty"], errors="coerce")
df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce")

conn = sqlite3.connect(":memory:")
df.to_sql("data", conn, index=False)

def q(sql):
    return pd.read_sql(sql, conn)

q("SELECT * FROM data LIMIT 5")


,transaction_id,transaction_date,transaction_time,transaction_qty,store_id,store_location,product_id,unit_price,product_category,product_type,product_detail
0,1,2023-01-01,07:06:11,2,5,Lower Manhattan,32,3.0,Coffee,Gourmet brewed coffee,Ethiopia Rg
1,2,2023-01-01,07:08:56,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg
2,3,2023-01-01,07:14:04,2,5,Lower Manhattan,59,4.5,Drinking Chocolate,Hot chocolate,Dark chocolate Lg
3,4,2023-01-01,07:20:24,1,5,Lower Manhattan,22,2.0,Coffee,Drip coffee,Our Old Time Diner Blend Sm
4,5,2023-01-01,07:22:41,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg


In [7]:

q("SELECT * FROM data LIMIT 5")



,transaction_id,transaction_date,transaction_time,transaction_qty,store_id,store_location,product_id,unit_price,product_category,product_type,product_detail
0,1,2023-01-01,07:06:11,2,5,Lower Manhattan,32,3.0,Coffee,Gourmet brewed coffee,Ethiopia Rg
1,2,2023-01-01,07:08:56,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg
2,3,2023-01-01,07:14:04,2,5,Lower Manhattan,59,4.5,Drinking Chocolate,Hot chocolate,Dark chocolate Lg
3,4,2023-01-01,07:20:24,1,5,Lower Manhattan,22,2.0,Coffee,Drip coffee,Our Old Time Diner Blend Sm
4,5,2023-01-01,07:22:41,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg


In [8]:

q("SELECT COUNT(*) AS total_rows FROM data")


,total_rows
0,149116


My table contains 149116 rows.One row represents one coffee shop transaction line item.Numbers to calculate: transaction_qty and unit_price (used together to calculate revenue).Groups to compare: product_category, product_type, and store_location.


SQL Research Questions
Q1: How many transaction records occurred at the Astoria coffee shop?Columns: store_location, transaction_idQ2: Which product category generated the highest total revenue?Columns: product_category, transaction_qty, unit_priceQ3: Which five product types generated the most revenue at the Astoria location?Columns: product_type, store_location, transaction_qty, unit_price

In [9]:
q("""
SELECT COUNT(*) AS total_records
FROM data
WHERE store_location = 'Astoria'
""")


,total_records
0,50599


The Astoria location recorded 50599 transaction line items.

In [10]:

q("""
SELECT product_category,
       ROUND(SUM(transaction_qty * unit_price), 2) AS total_revenue
FROM data
GROUP BY product_category
ORDER BY total_revenue DESC
""")


,product_category,total_revenue
0,Coffee,269952.45
1,Tea,196405.95
2,Bakery,82315.64
3,Drinking Chocolate,72416.00
4,Coffee beans,40085.25
5,Branded,13607.00
6,Loose Tea,11213.60
7,Flavours,8408.80
8,Packaged Chocolate,4407.64


In [11]:

q("""
SELECT product_type,
       ROUND(SUM(transaction_qty * unit_price), 2) AS total_revenue
FROM data
WHERE store_location = 'Astoria'
GROUP BY product_type
ORDER BY total_revenue DESC
LIMIT 5
""")



,product_type,total_revenue
0,Barista Espresso,27935.00
1,Brewed Chai tea,27427.90
2,Hot chocolate,26335.25
3,Gourmet brewed coffee,23823.30
4,Brewed herbal tea,16282.00


Coffee  generated the highest total revenue, earning $269952.